# 01 — Preprocessing Wajah (MTCNN + Alignment)**Tujuan:** mengubah foto mentah di `./data_riset/raw` menjadi **wajah ter-align 160×160** yang siap dipakai notebook 02 (augmentasi), 03 (enrollment), dan 06 (kamera live).**Prasyarat** (§6.2/§6.3 — sudah terpasang di laptop ini):```batpy -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126py -m pip install --no-cache-dir "opencv-python==4.14.0.94"py -m pip install "albumentations==1.4.16" pillow-heif scikit-learn matplotlib pillow numpypy -m pip install --no-deps facenet-pytorch==2.6.0```> `facenet-pytorch` **wajib `--no-deps`** — metadata 2.6.0 meng-pin `numpy<2` & `torch<2.3`, sehingga pip tries build numpy dari sumber dan gagal di Python 3.14. Sudah terverifikasi jalan di torch 2.14 + numpy 2.5.**Urutan eksekusi:** jalankan sel **SEL 1 → SEL 10** berurutan dari kernel kosong. Semua sel mandiri (tidak ada variabel dari notebook lain). Jika kernel di-restart, jalankan ulang mulai SEL 1.**Output (artefak untuk notebook berikutnya):**| Artefak | Isi ||---|---|| `./data_riset/aug\aligned\<nama_folder_orang>\g<NNNN>.jpg` | wajah ter-align, RGB, uint8, 160×160 || `./data_riset/aug\aligned_manifest.csv` | pemetaan **grup foto asli → file aligned** + status kualitas (dasar split anti-bocor §10.3) |**Aturan yang dijaga notebook ini (PRD §8.4 — pelajaran yang tidak boleh terulang):**1. Preprocessing training/enrollment **harus identik** dengan inferensi.2. Split nanti dilakukan **per grup foto asli**, bukan per file hasil augmentasi (anti data-leakage).3. HEIC ber-ekstensi `.jpg` harus terbaca (30 file di dataset).4. `mtcnn.detect(..., landmarks=True)` dan `mtcnn.extract(..., save_path=None)` wajib eksplisit.

In [ ]:
# SEL 1 — Impor paket + konstanta jalur (semua path diubah di sini saja)
import os
import sys
import time
import random
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import torch
import pillow_heif
from PIL import Image

# Fallback baca HEIC (PRD §8.4 no. 7): daftarkan opener PIL agar file .jpg
# yang sebenarnya HEIC (kamera iPhone) tetap bisa dibaca.
pillow_heif.register_heif_opener()

from facenet_pytorch import MTCNN
import matplotlib
import matplotlib.pyplot as plt

# --- struktur direktori (PRD §20.B) ---
DATA_ROOT = r"./data_riset/raw"          # dataset asli (sumber kebenaran)
MODEL_DIR = os.path.join(DATA_ROOT, "model")            # artefak baseline
AUG_ROOT = r"./data_riset/aug"        # dataset augmentasi
ALIGN_DIR = os.path.join(AUG_ROOT, "aligned")           # output notebook ini
KARANTINA_DIR = os.path.join(AUG_ROOT, "karantina")     # foto bermutu rendah, tidak dilath
MANIFEST_PATH = os.path.join(AUG_ROOT, "aligned_manifest.csv")

# --- parameter preprocessing (PRD §11) ---
IMG_SIZE = 160          # ukuran kanvas wajah (bukan 224)
CROP_FACTOR = 1.15      # luas bbox detektor yang dipakai (1.15 = "+15% di tiap sisi")
ROLL_CORRECT = True     # koreksi kemiringan (roll) dari garis kedua mata
MAX_SIDE = 1600         # downscale sisi terpanjang sebelum deteksi (hemat decode HEIC)
MIN_FOTOS_ORANG = 3     # syarat minimum foto asli per orang
SEED = 42
EXT_OK = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".heic", ".heif")
SKIP_DIRS = ("model", "__pycache__", ".ipynb_checkpoints", "aligned")

# --- reproducibility ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(ALIGN_DIR, exist_ok=True)
print("Python     :", sys.version.split()[0])
print("DATA_ROOT  :", DATA_ROOT, "| ada:", os.path.isdir(DATA_ROOT))
print("ALIGN_DIR  :", ALIGN_DIR, "| ada:", os.path.isdir(ALIGN_DIR))
print("DEVICE     :", DEVICE)

In [ ]:
# SEL 2 — Cek environment: GPU + versi paket (harus cocok PRD §6.2 & §8.5)
print("torch          :", torch.__version__)
print("torchvision    :", __import__("torchvision").__version__)
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            :", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))
    print("arch list      :", torch.cuda.get_arch_list())
    print("vram (GB)      :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("opencv         :", cv2.__version__)
print("numpy          :", np.__version__)
print("pandas         :", pd.__version__)
print("pillow_heif    :", pillow_heif.__version__)
print("facenet-pytorch: 2.6.0 (dipasang dengan --no-deps, lihat PRD §8.4 no. 6)")

# pemeriksaan cepat fungsi Haar/landmark yang pernah jadi sumber masalah
assert hasattr(cv2, "FaceDetectorYN"), "cv2_face_detection_yn tidak tersedia -> versi OpenCV salah"
assert hasattr(cv2, "CascadeClassifier"), "CascadeClassifier hilang -> OpenCV 5 terinstal (jangan)"
print("\nOK: prasyarat OpenCV untuk fallback DNN & Haar terpenuhi.")

In [ ]:
# SEL 3 — Pembaca gambaruniversal (aman path non-ASCII + fallback HEIC)
def imread_unicode(path):
    """Baca gambar dari path Windows mana pun. Return BGR uint8, atau None.
    1) coba OpenCV (paling cepat, aman untuk path non-ASCII)
    2) gagal -> PIL + pillow_heif (menangani HEIC yang menyamar jadi .jpg)
    """
    img = None
    try:
        buf = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    except Exception:
        img = None
    if img is None:
        try:
            with Image.open(path) as pil:
                rgb = pil.convert("RGB")
            img = cv2.cvtColor(np.array(rgb), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


def imwrite_unicode(path, img, quality=95):
    """Simpan JPG dengan nama file Unicode-safe (dipakai Windows)."""
    ext = os.path.splitext(path)[1]
    params = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)] if ext.lower() in (".jpg", ".jpeg") else []
    ok, buf = cv2.imencode(ext, img, params)
    if not ok:
        raise IOError("gagal encode: " + path)
    buf.tofile(path)


def is_heic(path):
    """Deteksi HEIC asli meskipun ekstensi file tertulis .jpg / .Jpg."""
    try:
        with open(path, "rb") as fh:
            return fh.read(12)[4:12] == b"ftypheic"
    except Exception:
        return False


# uji baca silang: hanya file gambar (ekstensi dikenal), termasuk HEIC tersembunyi
uji = []
for dp, dn, fn in os.walk(DATA_ROOT):
    dn[:] = [d for d in dn if d not in SKIP_DIRS]
    for f in sorted(fn):
        if os.path.splitext(f)[1].lower() in EXT_OK:
            uji.append(os.path.join(dp, f))
    if len(uji) > 80:
        break

heic_total = sum(1 for p in uji if is_heic(p))
gagal = []
for p in uji:
    if imread_unicode(p) is None:
        gagal.append(p)

print("file diuji          :", len(uji))
print("terdeteksi HEIC     :", heic_total, "(bukan error, akan diproses lewat pillow_heif)")
print("gagal dibaca        :", len(gagal))
for p in gagal[:5]:
    print("   -", p)
assert len(gagal) == 0, "ada file yang tidak terbaca -> perbaiki SEL 3 / pasang pillow-heif"
print("\nOK: seluruh file terbaca (termasuk HEIC).")

In [ ]:
# SEL 4 — Fungsi alignment wajah (KANONIK — salin fungsi ini ke notebook 03 & 06)
#         agar preprocessing training/enrollment identik dengan inferensi (PRD §11)
#
# PENTING (hasil eksperimen 7 Okt 2026, lihat PRD §8.4 no. 13):
#   Crop WAJIB memakai bounding box dari DETEKTOR MTCNN (mencakup dahi & dagu),
#   BUKAN bbox dari 5 landmark (mata+hidung+mulut). Bbox landmark +
#   margin 0.35 memotong dahi/dagu sehingga embedding rusak:
#   within-person similarity 0.47 vs 0.72, top-1 78,3% vs 98,6%.
#   Koreksi roll (kemiringan dari garis mata) memberi tambahan kecil dan konsisten.


def roll_degrees(landmarks5):
    """Sudut kemiringan wajah (derajat) dari garis kedua mata."""
    pts = np.asarray(landmarks5, dtype=np.float32).reshape(-1, 2)
    d = pts[1] - pts[0]
    return float(np.degrees(np.arctan2(float(d[1]), float(d[0]))))


def align_face(bgr, box, landmarks5, canvas=IMG_SIZE, crop_factor=CROP_FACTOR,
               roll_correct=ROLL_CORRECT):
    """Buat wajah ter-align 160x160 dari bbox detektor + 5 landmark.
    aligned rectified — dipakai untuk embedding VGGFace2."""
    x0, y0, x1, y1 = [float(v) for v in box]
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    side = max(x1 - x0, y1 - y0) * float(crop_factor)

    img = bgr
    if roll_correct:
        sudut = roll_degrees(landmarks5)
        if abs(sudut) > 0.5:
            M = cv2.getRotationMatrix2D((cx, cy), sudut, 1.0)
            img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

    h, w = img.shape[:2]
    sx1 = int(max(0, round(cx - side / 2)))
    sy1 = int(max(0, round(cy - side / 2)))
    sx2 = int(min(w, round(cx + side / 2)))
    sy2 = int(min(h, round(cy + side / 2)))
    if sx2 - sx1 < 8 or sy2 - sy1 < 8:
        return center_square_fallback(bgr, canvas)
    crop = img[sy1:sy2, sx1:sx2]
    return cv2.resize(crop, (canvas, canvas), interpolation=cv2.INTER_AREA)


def center_square_fallback(bgr, canvas=IMG_SIZE):
    """Fallback terakhir (PRD §11): potong tengah bujur sangkar lalu resize."""
    h, w = bgr.shape[:2]
    s = min(h, w)
    y0 = (h - s) // 2
    x0 = (w - s) // 2
    return cv2.resize(bgr[y0:y0 + s, x0:x0 + s], (canvas, canvas), interpolation=cv2.INTER_AREA)


def clahe_bright(bgr):
    """CLAHE pada channel L -> menaikkan kontras foto gelap tanpa mengubah geometri."""
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


def gamma_bright(bgr, g=0.5):
    """Koreksi gamma (g < 1 = terangkan) untuk foto yang sangat gelap."""
    lut = np.array([((i / 255.0) ** g) * 255.0 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(bgr, lut)


def pick_largest_face(boxes, landmarks):
    """Ambil wajah terbesar (bisa ada >1 wajah dalam 1 foto). Return (box, landmark, jumlah)."""
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    landmarks = np.asarray(landmarks, dtype=np.float32).reshape(-1, 5, 2)
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    idx = int(np.argmax(area))
    return boxes[idx], landmarks[idx], int(len(boxes))


print("Fungsi alignment siap.")
print("  - canvas        :", IMG_SIZE, "x", IMG_SIZE)
print("  - crop factor   :", CROP_FACTOR, "(1.15 = bbox detektor + 15% per sisi)")
print("  - koreksi roll  :", ROLL_CORRECT, "(derajat dari garis kedua mata)")
print("  - sumber bbox   : DETEKTOR MTCNN (bukan bbox 5 landmark)")

In [ ]:
# SEL 5 — Scan dataset + penomoran grup foto asli (dasar split anti-bocor §10.3)


def scan_dataset(root=DATA_ROOT):
    """Baca struktur Kapita/<folder_induk>/<folder_orang>/<foto> -> DataFrame.
    Identitas diambil dari NAMA FOLDER ORANG (bukan folder induk).
    grup = nomor urut file asli per orang (urutan sorted, deterministik)."""
    records = []
    for parent in sorted(os.listdir(root)):
        pdir = os.path.join(root, parent)
        if not os.path.isdir(pdir) or parent in SKIP_DIRS:
            continue
        subs = [s for s in sorted(os.listdir(pdir)) if os.path.isdir(os.path.join(pdir, s))]
        if subs:
            people = [(s, os.path.join(pdir, s)) for s in subs]
        else:
            people = [(parent, pdir)]
        for person, ppath in people:
            files = sorted([f for f in os.listdir(ppath)
                            if os.path.splitext(f)[1].lower() in EXT_OK])
            for i, f in enumerate(files):
                records.append({
                    "person": person,
                    "grup": "g{:04d}".format(i),
                    "nama_file": f,
                    "path": os.path.join(ppath, f),
                    "path_rel": os.path.relpath(ppath, root),
                })
    return pd.DataFrame(records)


df_src = scan_dataset()
jumlah_orang = df_src["person"].nunique()
per_orang = df_src.groupby("person").size()

print("folder orang      :", jumlah_orang)
print("foto asli (total) :", len(df_src))
print("grup per orang    : min {} | maks {} | rata-rata {:.1f}".format(
    int(per_orang.min()), int(per_orang.max()), per_orang.mean()))
print("\ndistribusi jumlah foto:")
print(per_orang.value_counts().sort_index().to_string())

terbuang = sorted(per_orang[per_orang < MIN_FOTOS_ORANG].index.tolist())
print("\nkelas dengan < {} foto asli (akan dibuang, PRD §7.4): {}".format(MIN_FOTOS_ORANG, terbuang))

df_src = df_src[~df_src["person"].isin(terbuang)].reset_index(drop=True)
print("\nsetelah membuang kelas bermasalah:")
print("  kelas/orang :", df_src["person"].nunique(), "(target PRD §7.3 = 69)")
print("  grup foto   :", len(df_src), "(target PRD §7.3 = 680)")
print("\ncontoh 5 baris:")
print(df_src.head(5).to_string(index=False))
assert df_src["person"].nunique() == 69, "jumlah kelas tidak sesuai PRD §7.3"
assert len(df_src) == 680, "jumlah grup foto asli tidak sesuai PRD §7.3"
assert not df_src.duplicated(subset=["person", "grup"]).any(), "grup foto asli duplikat"
print("\nOK: scan dataset sesuai PRD §7.3 dan §10.3 (grup unik per orang).")

In [ ]:
# SEL 6 — Jalankan preprocessing SEMUA foto asli (MTCNN -> align -> 160x160)
# Catatan: InceptionResnetV1 TIDAK dipakai di notebook ini (baru dipakai di notebook 03).
#          Yang dipakai sekarang hanya MTCNN untuk deteksi wajah + 5 landmark.
mtcnn = MTCNN(keep_all=True, select_largest=True, min_face_size=20, device=DEVICE)
# Model kedua hanya untuk percobaan ulang pada foto gelap/marginal:
# ambang deteksi lebih longgar + wajah lebih kecil yang boleh dideteksi.
mtcnn_longgar = MTCNN(keep_all=True, select_largest=True, min_face_size=10,
                      thresholds=[0.5, 0.6, 0.5], device=DEVICE)
FORCE_ALL = False     # True = proses ulang walau file output sudah ada


def preprocess_image(path, mtcnn_model, mtcnn_longgar_model):
    """Return (face_bgr_uint8, status, jumlah_wajah).
    Rantai percobaan (§11 + Mitigasi R2/R3):
      1) MTCNN standar pada foto apa adanya
      2) MTCNN pada versi CLAHE (untuk foto gelap)
      3) MTCNN longgar pada versi gamma (untuk foto sangat gelap)
      4) gagal -> potong tengah + resize, dicatat sebagai fallback di manifest
    CATATAN: alignment SELALU memakai foto ASLI (geometri identik), hanya
    landmark yang diambil dari versi yang Terryang."""
    img = imread_unicode(path)
    if img is None:
        return None, "gagal_baca", 0

    h, w = img.shape[:2]
    if max(h, w) > MAX_SIDE:
        scale = MAX_SIDE / float(max(h, w))
        img = cv2.resize(img, (int(round(w * scale)), int(round(h * scale))),
                         interpolation=cv2.INTER_AREA)

    percobaan = [
        ("mtcnn", img, mtcnn_model),
        ("mtcnn_clahe", clahe_bright(img), mtcnn_model),
        ("mtcnn_gamma", gamma_bright(img, 0.5), mtcnn_longgar_model),
    ]
    for nama, gambar, model in percobaan:
        rgb = cv2.cvtColor(gambar, cv2.COLOR_BGR2RGB)
        boxes, probs, points = model.detect(rgb, landmarks=True)
        if boxes is None or len(boxes) == 0 or points is None:
            continue
        box, lmk, n_face = pick_largest_face(boxes, points)
        face = align_face(img, box, lmk)       # alignment tetap pada foto asli
        if nama == "mtcnn" and n_face > 1:
            nama = "mtcnn_multi_wajah"
        return face, nama, n_face

    return center_square_fallback(img), "fallback_gambar_utuh", 0


rows = []
t_start = time.time()
total = len(df_src)
for i, rec in enumerate(df_src.itertuples(index=False), start=1):
    out_dir = os.path.join(ALIGN_DIR, rec.person)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, rec.grup + ".jpg")

    if (not FORCE_ALL) and os.path.exists(out_path):
        rows.append(dict(person=rec.person, grup=rec.grup, nama_file=rec.nama_file,
                         path_asli=rec.path, path_aligned=out_path,
                         status="skip_udah_ada", jumlah_wajah=0,
                         lebar_asli=0, tinggi_asli=0, heic=False))
    else:
        face, status, n_face = preprocess_image(rec.path, mtcnn, mtcnn_longgar)
        if face is not None and status == "fallback_gambar_utuh":
            # KARANTINA: MTCNN gagal total (umumnya potret samping / wajah tertutup).
            # Foto ini TIDAK boleh jadi data latih - dipindah ke folder karantina.
            k_dir = os.path.join(KARANTINA_DIR, rec.person)
            os.makedirs(k_dir, exist_ok=True)
            imwrite_unicode(os.path.join(k_dir, rec.grup + ".jpg"), face, quality=95)
            rows.append(dict(person=rec.person, grup=rec.grup, nama_file=rec.nama_file,
                             path_asli=rec.path, path_aligned="",
                             status="karantina_fallback", jumlah_wajah=0,
                             lebar_asli=im.shape[1], tinggi_asli=im.shape[0],
                             heic=is_heic(rec.path)))
            if i % 50 == 0 or i == total:
                elapsed = time.time() - t_start
                print("  [{}/{}] {:5.1f}%  elapsed {:.1f}s".format(
                    i, total, 100.0 * i / total, elapsed))
            continue
        if face is None:
            rows.append(dict(person=rec.person, grup=rec.grup, nama_file=rec.nama_file,
                             path_asli=rec.path, path_aligned="",
                             status=status, jumlah_wajah=0,
                             lebar_asli=0, tinggi_asli=0, heic=is_heic(rec.path)))
        else:
            imwrite_unicode(out_path, face, quality=95)
            im = imread_unicode(rec.path)
            rows.append(dict(person=rec.person, grup=rec.grup, nama_file=rec.nama_file,
                             path_asli=rec.path, path_aligned=out_path,
                             status=status, jumlah_wajah=n_face,
                             lebar_asli=im.shape[1], tinggi_asli=im.shape[0],
                             heic=is_heic(rec.path)))

    if i % 50 == 0 or i == total:
        elapsed = time.time() - t_start
        eta = elapsed / i * (total - i)
        print("  [{}/{}] {:5.1f}%  elapsed {:.1f}s  ETA {:.1f}s".format(
            i, total, 100.0 * i / total, elapsed, eta))

df_manifest = pd.DataFrame(rows)
df_manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\nselesai dalam {:.1f} detik ({:.3f} s/foto)".format(
    time.time() - t_start, (time.time() - t_start) / max(total, 1)))
print("manifest  :", MANIFEST_PATH)
print("\nrincian status:")
print(df_manifest["status"].value_counts().to_string())
print("\nfoto HEIC yang berhasil diproses:", int(df_manifest["heic"].sum()))

In [ ]:
# SEL 7 — Verifikasi: bentuk data, konsistensi, dan Artifact (§19.2 butir 6)
import glob

df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig")
berhasil = df[df["path_aligned"].astype(str).str.len() > 0].copy()

print("baris manifest        :", len(df))
print("wajah berhasil terproses :", len(berhasil), "/", len(df))
print("kelas/orang di manifest:", berhasil["person"].nunique())
print("grup foto asli unik    :", berhasil.groupby(["person", "grup"]).ngroups)

# 1) semua file output benar-benar ada, 160x160, uint8
# Contoh acak diambil dengan numpy RNG (seed tetap) supaya hasilnya reproducible
# dan tidak bergantung pada implementasi internal pandas.
rng = np.random.default_rng(SEED)
n_cek = min(25, len(berhasil))
cek_acak = berhasil.iloc[np.sort(rng.choice(len(berhasil), size=n_cek, replace=False))]
bad = []
for r in cek_acak.itertuples(index=False):
    im = imread_unicode(r.path_aligned)
    if im is None or im.shape[0] != IMG_SIZE or im.shape[1] != IMG_SIZE:
        bad.append((r.path_aligned, None if im is None else im.shape))
print("\ncek acak {} file output: jumlah rusak = {}".format(len(cek_acak), len(bad)))
for b in bad[:5]:
    print("   -", b)

# 2) jumlah file di disk harus cocok manifest
jumlah_file_disk = len(glob.glob(os.path.join(ALIGN_DIR, "*", "*.jpg")))
print("file .jpg di disk     :", jumlah_file_disk)

# 3) fallback = kualitas data yang perlu diawasi (PRD R2)
fallback = df[df["status"].astype(str).str.startswith("fallback")]
print("\nBERMASALAH (perlu ditinjau manual):", len(fallback), "foto")
if len(fallback):
    print(fallback[["person", "grup", "nama_file", "status"]].head(20).to_string(index=False))

# 4) foto multi-wajah (kemungkinan foto bersama) -> jangan dipakai enrollment
multi = df[df["jumlah_wajah"].astype(float) > 1]
print("\nfoto dengan >1 wajah:", len(multi))
if len(multi):
    print(multi[["person", "grup", "nama_file", "jumlah_wajah"]].head(10).to_string(index=False))

# 5) tidak ada kelas dengan kurang dari MIN_FOTOS_ORANG foto ter-align
per_orang_ok = berhasil.groupby("person").size()
print("\nkelas ter-minimal foto ter-align: min =", int(per_orang_ok.min()),
      "| jumlah kelas =", len(per_orang_ok))

assert len(bad) == 0, "ada file output yang rusak / salah ukuran"
assert jumlah_file_disk == len(berhasil), "jumlah file disk tidak cocok manifest"
assert len(per_orang_ok) == 69, "harus ada 69 kelas ter-align"
assert int(per_orang_ok.min()) >= MIN_FOTOS_ORANG, "ada kelas dengan < 3 foto ter-align"
print("\nOK: semua pemeriksaan SEL 7 lolos.")

In [ ]:
# SEL 8 — Visualisasi: wajah ter-align vs foto asli (matplotlib)
def plot_grid(items, title, cols=6, suptitle=""):
    rows = int(np.ceil(len(items) / float(cols)))
    fig, axes = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, it in zip(axes, items):
        ax.imshow(cv2.cvtColor(it["img"], cv2.COLOR_BGR2RGB))
        ax.set_title(it["judul"], fontsize=7)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    fig.suptitle(suptitle + "\n" + title, fontsize=10)
    plt.tight_layout()
    plt.show()


# (a) 12 wajah ter-align dari orang berbeda
items = []
for person in sorted(berhasil["person"].unique())[:12]:
    sub = berhasil[berhasil["person"] == person]
    r = sub.iloc[0]
    im = imread_unicode(r.path_aligned)
    items.append({"img": im, "judul": "{}\n{} {}".format(person.split("_")[0], r.grup, r.status)})
plot_grid(items, "Wajah ter-align 160x160 (MTCNN + crop factor {} + roll)".format(CROP_FACTOR),
          suptitle="(a) Sampel wajah ter-align")

# (b) foto asli + bbox MTCNN -> wajah ter-align (3 contoh, termasuk 1 fallback bila ada)
def gambar_asli_dan_bbox(path):
    """Kembalikan salinan foto asli dengan kotak wajah & 5 landmark digambar."""
    img = imread_unicode(path)
    vis = img.copy()
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIDE:
        s = MAX_SIDE / float(max(h, w))
        vis = cv2.resize(vis, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    boxes, probs, points = mtcnn.detect(rgb, landmarks=True)
    if boxes is not None and len(boxes) > 0:
        for bx in np.asarray(boxes, dtype=np.float32):
            x0, y0, x1, y1 = [int(round(v)) for v in bx]
            cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 255, 0), 3)
        if points is not None:
            for p in np.asarray(points, dtype=np.float32).reshape(-1, 5, 2):
                for px, py in p:
                    cv2.circle(vis, (int(round(px)), int(round(py))), 3, (0, 0, 255), -1)
    return vis


sampled = berhasil.iloc[np.sort(rng.choice(len(berhasil), size=3, replace=False))]
items2 = []
for r in sampled.itertuples(index=False):
    nama = r.person.split("_")[0]
    items2.append({"img": gambar_asli_dan_bbox(r.path_asli),
                   "judul": "ASLI + bbox/landmark\n{} {}".format(nama, r.grup)})
    items2.append({"img": imread_unicode(r.path_aligned),
                   "judul": "TER-ALIGN {}x{}\n{} {}".format(IMG_SIZE, IMG_SIZE, nama, r.status)})
plot_grid(items2, "Foto asli (hijau = bbox MTCNN, merah = 5 landmark) vs hasil alignment",
          cols=6, suptitle="(b) Pemetaan grup foto asli -> g<N>.jpg")

print("Total file ter-align :", len(berhasil))
print("Total kelas/orang   :", berhasil['person'].nunique())
print("Lokasi              :", ALIGN_DIR)
print("Manifest            :", MANIFEST_PATH)

# SEL 9 — Ringkasan & langkah berikutnya**Yang sudah tercapai pada notebook ini:**| Target PRD | Hasil ||---|---|| §11 preprocessing identik (MTCNN → crop bbox detektor → roll → 160) | terpasang di SEL 4, fungsi kanonik untuk notebook 03 & 06 || §7.3 dataset 69 kelas / 680 grup | terverifikasi lewat `assert` di SEL 5 || §10.3 grup foto asli sebagai unit split | kolom `person` + `grup` + `aligned_manifest.csv` || §8.4 no. 7 file HEIC | terbaca via pillow_heif (30 file) || §8.4 no. 8 API MTCNN | `landmarks=True` & `save_path=None` eksplisit |**Normalisasi `(pixel - 127.5) / 128` sengaja TIDAK dilakukan di sini** — yang disimpan ke disk adalah uint8 160×160 agar hemat ruang dan bisa divisualisasi. Normalisasi dikerjakan di notebook 03 (enrollment) dan 06 (live) memakai fungsi yang sama.**Langkah berikutnya (notebook 02):** baca `aligned_manifest.csv`, augmentasi tiap wajah ter-align dengan Albumentations (`N_AUG=25`, pipeline §10.2), simpan ke `Kapita_aug/augmented/<person>/<grup>_aug<i>.jpg` dengan `SEED=42`.> **Bila SEL 6/RSUME ada baris `fallback_gambar_utuh`**: periksa foto-foto tersebut secara manual. Bila memang salah framing (bukan wajah), hapus `g<N>.jpg` + baris manifest-nya agar tidak ikut ter-enrollment.

In [ ]:
# SEL 10 — Ringkasan akhir + pemeriksaan cepat tanpa perlu komputasi ulang
if "df_manifest" not in globals():
    df_manifest = pd.DataFrame()
ringkas = {
    "waktu_eksekusi_notebook": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "img_size": IMG_SIZE,
    "crop_factor": CROP_FACTOR,
    "max_side": MAX_SIDE,
    "seed": SEED,
    "device": str(DEVICE),
    "jumlah_orang": int(df_manifest["person"].nunique()),
    "jumlah_grup_asli": int(len(df_manifest)),
    "jumlah_wajah_berhasil": int((df_manifest["path_aligned"].astype(str).str.len() > 0).sum()),
    "jumlah_fallback": int(df_manifest["status"].astype(str).str.startswith("fallback").sum()),
    "status_counts": {str(k): int(v) for k, v in df_manifest["status"].value_counts().items()},
}
for k, v in ringkas.items():
    print("{:<26}: {}".format(k, v))

print("\nArtefak untuk notebook 02:")
print("  1. folder wajah aligned :", ALIGN_DIR)
print("  2. manifest           :", MANIFEST_PATH)
print("\nLanjut -> 02_augmentasi_massal.ipynb")

In [ ]:
from pathlib import Path
import nbformat
from copy import deepcopy

# Folder tempat notebook berada
folder = Path.cwd()

# Urutan notebook yang akan digabungkan
daftar_notebook = [
    "01_preprocessing_wajah.ipynb",
    "02_augmentasi_massal.ipynb",
    "03_enrollment_gallery.ipynb",
    "04_kalibrasi_threshold.ipynb",
    "05_evaluasi.ipynb",
    "06_live_absensi.ipynb",
]

# File output baru
notebook_gabungan = nbformat.v4.new_notebook()
notebook_gabungan.cells = []

for nama_file in daftar_notebook:
    path_file = folder / nama_file

    if not path_file.exists():
        raise FileNotFoundError(
            f"Notebook tidak ditemukan: {path_file}"
        )

    nb = nbformat.read(path_file, as_version=4)

    # Penanda bagian agar asal setiap cell jelas
    notebook_gabungan.cells.append(
        nbformat.v4.new_markdown_cell(
            f"# Bagian: {nama_file}"
        )
    )

    # Salin seluruh cell tanpa mengubah file sumber
    notebook_gabungan.cells.extend(
        deepcopy(nb.cells)
    )

# Gunakan metadata kernel dari notebook pertama
nb_pertama = nbformat.read(
    folder / daftar_notebook[0],
    as_version=4
)
notebook_gabungan.metadata = deepcopy(nb_pertama.metadata)

# Validasi struktur notebook
nbformat.validate(notebook_gabungan)

# Simpan sebagai file baru
output = folder / "kapitaselekta_gabungan.ipynb"
nbformat.write(notebook_gabungan, output)

print("Penggabungan selesai!")
print(f"Lokasi file: {output}")
print(f"Jumlah notebook: {len(daftar_notebook)}")
print(f"Jumlah cell: {len(notebook_gabungan.cells)}")
